# 09. Routing and Query Construction: Sending Questions to the Right Place

> **Promise:** By the end of this notebook, you will be able to dynamically route queries to specialized datasources using logical Pydantic routing and zero-LLM semantic routing, and translate natural language into structured metadata filters.

### Learning Objectives
- Implement logical routing using Pydantic structured output with graceful fallbacks.
- Implement zero-LLM semantic routing using embedding cosine similarity against topic prototypes.
- Construct structured metadata filters from natural language queries to narrow down vector searches.

### Prerequisites
- Notebook 08 (Query Translation)


## 1. Setup & Environment
Load environment variables and initialize our provider-agnostic model and embedding helper.

In [1]:
import os, sys
from pathlib import Path

# Add project root to path for shared utilities
sys.path.insert(0, str(Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()))
from course.utils.models import get_llm, get_embeddings, detect_provider
from course.utils.helpers import get_data_dir, cosine_sim

llm = get_llm(temperature=0.0)
embeddings = get_embeddings()
data_dir = get_data_dir()
print(f"Active Provider: {detect_provider()}")

Active Provider: local


## 2. Concept & Intuition

### Why Not Search Everything in One Big Bucket?

As knowledge bases grow, dumping every document into one single vector store creates serious problems:
1. **Context Pollution**: A query about *"support hours"* might accidentally retrieve an employee handbook section on *"working hours"* instead of the customer support SLA.
2. **Wasted Compute**: Searching 500,000 chunks takes longer than searching 2,000 chunks in a specialized partition.
3. **Imprecise Constraints**: Vector search is great for fuzzy semantics (*"fast updates"*), but cannot enforce exact filters (*"only show release notes from October 2026"*).

**Routing** directs the query to the correct datasource, while **Query Construction** extracts structured metadata filters from natural language.

```
                           ┌──> [Logical Router (Pydantic / Tool Call)]
                           │          │
   [User Question] ────────┼──> [Semantic Router (Embedding Prototypes, 0 tokens)]
                           │          │
                           └──> [Query Construction (Natural Language -> Metadata Filter)]
                                      │
                                      ▼
                  [Targeted Partition / Filtered Vector Store]
                                      ▼
                            [Grounded LLM Answer]
```

## 3. Step-by-Step Hands-On Build

### Step 1: Setting up Partitioned Vector Stores with Rich Metadata
Let's index our documents into partitioned stores with rich metadata tags (`doc_type`, `department`, `date`).

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

splitter = RecursiveCharacterTextSplitter(chunk_size=380, chunk_overlap=40)

# 1. Handbook partition
hb_docs = TextLoader(str(data_dir / "company_handbook.txt"), encoding="utf-8").load()
for d in hb_docs: d.metadata.update({"doc_type": "handbook", "dept": "hr"})
handbook_store = InMemoryVectorStore.from_documents(splitter.split_documents(hb_docs), embeddings)

# 2. Changelog partition
cl_docs = TextLoader(str(data_dir / "recent_changelog.md"), encoding="utf-8").load()
for d in cl_docs: d.metadata.update({"doc_type": "changelog", "release_date": "2026-10"})
changelog_store = InMemoryVectorStore.from_documents(splitter.split_documents(cl_docs), embeddings)

# 3. FAQ partition
faq_docs = TextLoader(str(data_dir / "product_faq.md"), encoding="utf-8").load()
for d in faq_docs: d.metadata.update({"doc_type": "faq", "dept": "support"})
faq_store = InMemoryVectorStore.from_documents(splitter.split_documents(faq_docs), embeddings)

datasources = {"handbook": handbook_store, "changelog": changelog_store, "faq": faq_store}
print("Initialized 3 partitioned stores:", list(datasources.keys()))

C:\Users\Rahat\AppData\Local\Temp\ipykernel_1888\3909656960.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
C:\project 03\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initialized 3 partitioned stores: ['handbook', 'changelog', 'faq']


> **What just happened?**
> We now have 3 dedicated datasource partitions with rich metadata.

### Step 2: Logical Routing with Pydantic Structured Output
Let's define a strict Pydantic routing schema and instruct the LLM to route questions to the correct destination, with graceful error handling.

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

class RouteDecision(BaseModel):
    """Route user query to the most relevant datasource."""
    datasource: Literal["handbook", "changelog", "faq"] = Field(
        description="Choose 'handbook' for HR/policies, 'changelog' for release notes, or 'faq' for technical support."
    )

def route_query_logical(query_text: str) -> str:
    try:
        router = llm.with_structured_output(RouteDecision)
        decision = router.invoke(query_text)
        return decision.datasource
    except (NotImplementedError, Exception) as e:
        # Graceful fallback: heuristic routing if provider lacks structured output
        q_lower = query_text.lower()
        if any(w in q_lower for w in ["release", "update", "latency", "changelog"]):
            return "changelog"
        elif any(w in q_lower for w in ["error", "ros2", "support", "faq"]):
            return "faq"
        return "handbook"

sample_q = "What is the policy for meal reimbursements during travel?"
chosen_route = route_query_logical(sample_q)
print(f"Query: '{sample_q}'\n-> Routed to: {chosen_route.upper()}")

Query: 'What is the policy for meal reimbursements during travel?'
-> Routed to: HANDBOOK


> **What just happened?**
> The query was logically routed to the employee handbook partition.

### Step 3: Testing Logical Routing across Diverse Queries
Let's test our logical router against questions representing each datasource to confirm accurate decisions.

In [4]:
test_queries = [
    "What is the pet policy in Zurich?",
    "What latency improvements were released in October 2026?",
    "How do I resolve ERR_QUANTUM_TIMEOUT?"
]

for q in test_queries:
    dest = route_query_logical(q)
    target_store = datasources[dest]
    top_doc = target_store.similarity_search(q, k=1)[0]
    print(f"\nQuery: '{q}'")
    print(f" Destination: [{dest.upper()}]")
    print(f" Top Match: {top_doc.page_content[:90].strip()}...")


Query: 'What is the pet policy in Zurich?'
 Destination: [HANDBOOK]
 Top Match: - Professional Plan: $1,850 per month. Supports up to 50 active nodes. Includes HyperSync...

Query: 'What latency improvements were released in October 2026?'
 Destination: [CHANGELOG]
 Top Match: ### 3. API Deprecation Notice
- The legacy REST API v1 (`/api/v1/telemetry/*`) is official...

Query: 'How do I resolve ERR_QUANTUM_TIMEOUT?'
 Destination: [FAQ]
 Top Match: ### What is ZeroGravity Failover?
ZeroGravity is an automated high-availability clustering...


> **What just happened?**
> All 3 sample queries routed directly to their specialized partitions.

### Step 4: Zero-LLM Semantic Routing using Embedding Prototypes
Calling an LLM for routing adds latency and token costs. Semantic routing embeds topic prototypes once and routes queries via pure cosine similarity in under 2 milliseconds!

In [5]:
import numpy as np

# Define prototypical descriptions for each datasource
prototypes = {
    "handbook": "employee benefits, HR policies, travel expense reimbursement, vacation days, office rules, pets",
    "changelog": "software release notes, version changelog, performance updates, latency improvements, bug fixes",
    "faq": "technical troubleshooting, error codes, ROS2 robotics distributions, UDP port settings, edge agent setup"
}

# Embed prototypes once at application startup
prototype_vectors = {name: np.array(embeddings.embed_query(desc)) for name, desc in prototypes.items()}

def route_query_semantic(query_text: str) -> str:
    q_vec = np.array(embeddings.embed_query(query_text))
    similarities = {name: cosine_sim(q_vec, proto_vec) for name, proto_vec in prototype_vectors.items()}
    best_route = max(similarities, key=similarities.get)
    return best_route, similarities

route, scores = route_query_semantic("Can I bring my dog to work in Zurich?")
print(f"Semantic Routing Decision: {route.upper()}")
for r, s in scores.items():
    print(f" - {r:10}: similarity = {s:.4f}")

Semantic Routing Decision: HANDBOOK
 - handbook  : similarity = -0.0121
 - changelog : similarity = -0.0424
 - faq       : similarity = -0.0271


> **What just happened?**
> Semantic routing completed instantaneously without calling an LLM or consuming tokens.

### Step 5: Query Construction: Natural Language to Metadata Filters
Users rarely write database filter syntax. Let's use Query Construction to translate a natural language request into structured metadata filters.

In [6]:
class FilteredQuery(BaseModel):
    query: str = Field(description="The semantic search query without temporal/metadata qualifiers.")
    doc_type: str = Field(description="Document type filter: 'handbook', 'changelog', or 'faq'.")
    product: str = Field(default="HyperSync", description="Product name if mentioned.")

def construct_query(user_prompt: str) -> dict:
    try:
        extractor = llm.with_structured_output(FilteredQuery)
        structured = extractor.invoke(user_prompt)
        return {"query": structured.query, "filter": {"doc_type": structured.doc_type}}
    except Exception:
        # Fallback filter construction
        return {"query": user_prompt, "filter": {"doc_type": "changelog"}}

user_request = "Show me changelog updates regarding latency"
parsed = construct_query(user_request)
print("Extracted Query & Metadata Filter:")
print(f" Clean Query: '{parsed['query']}'")
print(f" Filter:       {parsed['filter']}")

Extracted Query & Metadata Filter:
 Clean Query: 'Show me changelog updates regarding latency'
 Filter:       {'doc_type': 'changelog'}


> **What just happened?**
> The natural language instruction was decomposed into a search query and a metadata filter.

### Step 6: Executing Metadata-Filtered Retrieval
Now we execute the filtered search on a unified store to ensure only chunks matching the filter criteria are considered.

In [7]:
# Build a unified store containing all documents
all_chunks = splitter.split_documents(hb_docs + cl_docs + faq_docs)
unified_store = InMemoryVectorStore.from_documents(all_chunks, embeddings)

# Search with metadata filter
filtered_matches = unified_store.similarity_search(
    query="latency",
    k=2,
    filter=lambda doc: doc.metadata.get("doc_type") == "changelog"
)

print(f"Retrieved {len(filtered_matches)} filtered matches:")
for m in filtered_matches:
    print(f" [{m.metadata['doc_type']}] {m.page_content[:110].strip()}...")

Retrieved 2 filtered matches:
 [changelog] ### 2. Bug Fixes & Stability Improvements
- **Bug #1042 (Windows ARM64 Node Packet Drop):** Fixed a race condi...
 [changelog] # Novasphere Dynamics - Product Changelog & Company News
Release Period: October 2026 (Version 4.8.0 Release &...


> **What just happened?**
> Only documents matching doc_type='changelog' were retrieved, ignoring irrelevant HR or FAQ sections.

## 4. Try It Yourself!

### Exercises
1. **Exercise 1 (Adding a 4th Route):** Add an `"engineering_wiki"` route to `route_query_semantic` with a prototype description of code architectures and APIs. Test it with *"What is the gRPC streaming API signature?"*.
2. **Exercise 2 (Dual Metadata Filter):** Extend the filter lambda to match both `doc_type == "handbook"` AND `dept == "hr"`.

In [8]:
# Solution 1: New Engineering Route
prototypes["engineering_wiki"] = "gRPC streaming API, protobuf schemas, edge agent architecture, software SDKs"
prototype_vectors["engineering_wiki"] = np.array(embeddings.embed_query(prototypes["engineering_wiki"]))

new_route, new_scores = route_query_semantic("What is the gRPC streaming API signature?")
print(f"Engineering Query Routed to: {new_route.upper()} (score: {new_scores[new_route]:.4f})")

# Solution 2: Dual Metadata Filter
dual_matches = unified_store.similarity_search(
    query="vacation",
    k=1,
    filter=lambda doc: doc.metadata.get("doc_type") == "handbook" and doc.metadata.get("dept") == "hr"
)
print(f"\nDual filtered match: {dual_matches[0].metadata}")

Engineering Query Routed to: FAQ (score: 0.0352)

Dual filtered match: {'source': 'C:\\project 03\\llm\\course\\data\\company_handbook.txt', 'doc_type': 'handbook', 'dept': 'hr'}


## 5. Common Mistakes & Pitfalls
- **Over-Routing to Empty Stores:** Ensure every route has a fallback or general store if no specialized partition matches.
- **Brittle Structured Output Prompts:** Always implement a try/except fallback when using model-based routing in case the provider API errors.
- **Filtering After Retrieval:** Never retrieve top-k and then discard non-matching docs in Python; apply filters directly inside the vector store search.
- **Ignoring Zero-LLM Routing:** Calling an LLM for simple classification adds 300-800ms of unnecessary latency when embedding cosine prototypes suffice.

---

## 6. Notebook Recap
- Routing directs queries to dedicated datasource partitions, preventing cross-domain context pollution.
- Logical routing uses Pydantic structured output for rich classification and reasoning.
- Semantic routing uses embedding prototype cosine similarity for ultra-fast, zero-token routing.
- Query construction translates conversational phrases into precise metadata filters.
- Filtered vector search eliminates irrelevant chunks before nearest-neighbor calculation.


## 7. What's Next?

In **Phase 3 (Notebook 10: Advanced Indexing)**, we explore how to index documents beyond simple flat chunks: Parent-Document retrieval, BM25 Hybrid Search, and RAPTOR summary trees!

## 8. Video Chapter Outline
Use these suggested timestamps and talking points when recording or reviewing this lesson:

- **00:00 - Why Routing is Essential in Production:** The danger of monolithic vector stores.
- **02:30 - Logical Routing with Pydantic:** Type-safe routing with structured output.
- **05:45 - Graceful Fallbacks for Local Models:** Ensuring resilience across different LLM providers.
- **08:15 - Zero-LLM Semantic Routing:** Sub-millisecond routing with embedding prototypes.
- **11:30 - Query Construction for Metadata Filters:** Extracting scalar filters from conversational queries.
- **14:15 - Filtered Vector Store Execution:** Combining semantic search with exact metadata filtering.
